# Download Data

This example notebook shows you how to download water data time-series from Google Earthengine for
1. Annual JRC surface water data
2. Dynamic World - monthly aggregated data
3. Dynamic World - per acquisition (unaggregated) data
4. OPERA DSWx-S1 - radar surface water, per acquisition and monthly

### Imports

In [ ]:
from water_timeseries.downloader import EarthEngineDownloader
import geopandas as gpd

#### Loading vector dataset
* we are loading the provided lake polygon dataset
* the specific unique lake id is "id_geohash"
* explore the dataset

In [ ]:
vector_path = "../tests/data/lake_polygons.parquet"
gdf = gpd.read_parquet(vector_path)

In [ ]:
gdf.head()

In [ ]:
gdf.explore(width=1200, height=600)

#### Setup logger (optional)
* here I am using loguru, you can also choose others

In [ ]:
import loguru

logger = loguru.logger
logger.add("logfile_dl.log", level=0)

#### Setup GEE account
* set project name to EE_PROJECT environment variable (preferred)

#### Instantiate the downloader
* one downloader serves every product below - the calls differ only in the method name
* two ways to name the project, pick one: the `EE_PROJECT` environment variable (option 1, preferred,
  set in the cell below) or an explicit `ee_project=` argument (option 2, the commented line)
* pass a `logger` to get progress and per-chunk diagnostics


In [ ]:
import os

# Option 1 (preferred): set the project name as an environment variable -
# the downloader picks it up when no ee_project is passed
os.environ["EE_PROJECT"] = "YOUR_GEE_PROJECT"

In [ ]:
dl = EarthEngineDownloader(ee_auth=True, logger=logger)

# Option 2: skip the environment variable and pass the project id directly
# dl = EarthEngineDownloader(ee_project="YOUR_GEE_PROJECT", ee_auth=True, logger=logger)

#### Download JRC dataset
* requires polygon vector dataset + GEE account
* returns xarray dataset

#### Simple case JRC
* Download JRC annual timeseries for
  * default years (2000-2021)
  * all polygons
  * serially (if too large, polygon dataset will be chunked to avoid GEE Errors)
  * takes ~ 20 seconds
* This returns a xr.Dataset with
  * 2 coordinates: date, id_geohash (or your specific selection)
  * 4 variables: area_land, area_nodata, area_water_permanent, area_water_seasonal      

In [ ]:
ds_jrc = dl.download_jrc_annual(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    n_parallel=1,
)

In [ ]:
ds_jrc

#### Simple case Dynamic World
* Download Dynamic World monthly timeseries for
  * default years (2016-2025)
  * all polygons
  * serially (if too large, polygon dataset will be chunked to avoid GEE Errors)
  * takes ~ 20 seconds
* This returns a xr.Dataset with
  * 2 coordinates: date, id_geohash (or your specific selection)
  * 4 variables: bare, built, crops, flooded_vegetation, grass, shrub_and_scrub, snow_and_ice, trees, water

In [ ]:
ds_dw = dl.download_dw_monthly(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    years=range(2017, 2026),
    months=[6, 7, 8, 9],
    n_parallel=1,
)

In [ ]:
ds_dw

#### Simple case Dynamic World - per acquisition (daily)
* `download_dw_daily` returns one row per Sentinel-2 acquisition instead of the monthly mode composite
* it is **not** a gap-filled daily series: a lake gets roughly 20-50 observations per May-September season,
  and a missing date is a cloud/coverage gap rather than a download failure
* the date arguments are the same as for the monthly product, so the call above is repointed by changing the
  method name - or give a recurring within-year window with `start_date` / `end_date` as `MM-DD`
* an observation is masked when more than `max_nodata` (default 2 %) of the polygon has no data, and fully
  masked acquisitions are dropped: "no observation" is not "zero water"
* two seasons for all 118 polygons take ~1-2 minutes
* returns an `xr.Dataset` with the same 9 class variables as the monthly product, but one row per acquisition

In [ ]:
ds_dw_daily = dl.download_dw_daily(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    years=[2022, 2023],
    start_date="05-01",  # 'MM-DD', inclusive, applied to every year
    end_date="09-30",    # 'MM-DD', inclusive
    max_nodata=0.02,
)

In [ ]:
ds_dw_daily

#### OPERA DSWx-S1 - radar surface water
* Sentinel-1 based, 30 m, three classes: `area_open_water` (WTR 1), `area_inundated_vegetation` (WTR 3) and
  `area_not_water` (WTR 0)
* the date arguments are identical to Dynamic World's, so `download_dw_*` calls are repointed by changing the
  method name
* three limits matter before reading a series:
  * the archive starts in **2024-08** - this is a forward-looking supplement, never a 2016-forward source
  * water bodies below ~3 ha / 200 m width are **not mapped**, so sub-MMU polygons read as "no water" by design
  * there is **no snow/ice class** - a frozen basin is classified as inundated vegetation, so winter rows are an
    ice signal that has to be gated against Dynamic World's `snow_and_ice`
* downloads below use three lakes from the fixture (377, 207 and 127 ha) to keep the example short - the DSWx
  archive is denser than DW's and every polygon costs Earth Engine time

In [ ]:
dswx_lakes = ["b7g4rf3n3x43", "b7g6d4zhynr0", "b7g678chrqb5"]

ds_dswx_daily = dl.download_dswx_acquisitions(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    id_list=dswx_lakes,
    years=[2025],
    months=[6, 7, 8, 9],   # same argument as the monthly call below
)

In [ ]:
ds_dswx_daily

##### OPERA DSWx-S1 - monthly
* one per-pixel mode composite per month, the same statistic `download_dw_monthly` uses
* a month without a usable acquisition is **absent**, not zero
* overlapping MGRS tiles are collapsed to one row per (polygon, date), never summed

In [ ]:
ds_dswx_monthly = dl.download_dswx_monthly(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    id_list=dswx_lakes,
    years=[2025],
    months=[6, 7, 8, 9],
)

In [ ]:
ds_dswx_monthly